In [ ]:
import json
from pathlib import Path
from datasets import Dataset
from setfit import SetFitModel, SetFitTrainer
import torch
import numpy as np
import sys

# === Paths ===
DATA_PATH = Path(r"E:\Product Comparator\aspect_analysis\final_aspa_data_clustered.json")
MODEL_PATH = Path(r"E:\Product Comparator\model_aspa")
SAVE_PATH = Path(r"E:\Product Comparator\model_aspa_full")

# === Load dataset ===
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

dataset = Dataset.from_list([{"text": d["input"], "label": d["label"]} for d in data])

# === Encode labels as multi-hot vectors ===
all_labels = sorted({lbl for d in data for lbl in d["label"]})
label2id = {l: i for i, l in enumerate(all_labels)}
id2label = {i: l for l, i in label2id.items()}

def encode_labels(example):
    y = [0] * len(all_labels)
    for l in example["label"]:
        if l in label2id:
            y[label2id[l]] = 1
    example["label"] = y
    return example

dataset = dataset.map(encode_labels, batched=False, disable_nullable=True)


e:\Product Comparator\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 6917/6917 [00:19<00:00, 351.23 examples/s]


In [2]:
# === Shuffle and split ===
dataset = dataset.shuffle(seed=42)
train_ds, test_ds = dataset.train_test_split(test_size=0.1).values()

# === Load backbone ===
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.empty_cache()
model = SetFitModel.from_pretrained(MODEL_PATH).to(device)
print("✅ Loaded backbone from checkpoint.\n")

# === Precompute embeddings to save RAM ===

No sentence-transformers model found with name E:\Product Comparator\model_aspa. Creating a new one with mean pooling.
model_head.pkl not found in E:\Product Comparator\model_aspa, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


✅ Loaded backbone from checkpoint.



In [5]:
print("🔄 Precomputing embeddings for train and test sets...")
X_train = model.encode(train_ds["text"])
y_train = np.array(train_ds["label"])

🔄 Precomputing embeddings for train and test sets...


In [6]:
X_test = model.encode(test_ds["text"])
y_test = np.array(test_ds["label"])
print("✅ Embeddings precomputed.\n")

# === Use a lightweight sklearn head for training ===
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# Create binary classifier for multi-label


✅ Embeddings precomputed.



In [9]:
import numpy as np
from sklearn.linear_model import SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from scipy.sparse import csr_matrix

# Convert X_train and X_test to float32 to reduce memory
X_train = np.array(X_train, dtype=np.float32)
X_test = np.array(X_test, dtype=np.float32)

# Convert y_train and y_test to sparse matrices
y_train_sparse = csr_matrix(y_train)
y_test_sparse = csr_matrix(y_test)

# === Parameters ===
n_labels = y_train.shape[1]
chunk_size = 1000  # train 1000 labels at a time
trained_heads = []

for start in range(0, n_labels, chunk_size):
    end = min(start + chunk_size, n_labels)
    print(f"🚀 Training head for labels {start} to {end-1}...")
    
    y_chunk = y_train_sparse[:, start:end].toarray()  # convert to dense for sklearn
    head = OneVsRestClassifier(
        SGDClassifier(loss="log_loss", max_iter=1000, tol=1e-3, random_state=42)
    )
    head.fit(X_train, y_chunk)
    trained_heads.append(head)
    print(f"✅ Finished training labels {start} to {end-1}\n")

# === Evaluate in chunks ===
all_preds = []
for i, head in enumerate(trained_heads):
    start = i * chunk_size
    end = min((i+1)*chunk_size, n_labels)
    print(f"🔍 Predicting labels {start} to {end-1}...")
    y_chunk_pred = head.predict(X_test)
    all_preds.append(y_chunk_pred)

y_pred_full = np.hstack(all_preds)  # combine predictions


🚀 Training head for labels 0 to 999...


e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 17 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 38 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 50 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 51 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 79 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 86 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 

✅ Finished training labels 0 to 999

🚀 Training head for labels 1000 to 1999...


e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 31 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 60 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 68 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 72 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 73 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 103 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not

✅ Finished training labels 1000 to 1999

🚀 Training head for labels 2000 to 2999...


...

✅ Finished training labels 13000 to 13999

🚀 Training head for labels 14000 to 14999...


e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 32 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 35 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 48 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 54 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 56 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 57 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 

✅ Finished training labels 14000 to 14999

🚀 Training head for labels 15000 to 15791...


e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 8 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 42 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 45 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 47 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 50 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 83 is present in all training examples.
  warnings.warn(
e:\Product Comparator\venv\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 9

✅ Finished training labels 15000 to 15791

🔍 Predicting labels 0 to 999...
🔍 Predicting labels 1000 to 1999...
🔍 Predicting labels 2000 to 2999...
🔍 Predicting labels 3000 to 3999...
🔍 Predicting labels 4000 to 4999...
🔍 Predicting labels 5000 to 5999...
🔍 Predicting labels 6000 to 6999...
🔍 Predicting labels 7000 to 7999...
🔍 Predicting labels 8000 to 8999...
🔍 Predicting labels 9000 to 9999...
🔍 Predicting labels 10000 to 10999...
🔍 Predicting labels 11000 to 11999...
🔍 Predicting labels 12000 to 12999...
🔍 Predicting labels 13000 to 13999...
🔍 Predicting labels 14000 to 14999...
🔍 Predicting labels 15000 to 15791...


In [10]:
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

# y_test_sparse is sparse, convert to dense for metrics
y_test_dense = y_test_sparse.toarray()  

# y_pred_full is already dense after np.hstack
y_pred_dense = y_pred_full  

# === Multi-label metrics ===
print("🔍 Evaluation metrics:")

f1_micro = f1_score(y_test_dense, y_pred_dense, average="micro")
f1_macro = f1_score(y_test_dense, y_pred_dense, average="macro")

precision_micro = precision_score(y_test_dense, y_pred_dense, average="micro")
recall_micro = recall_score(y_test_dense, y_pred_dense, average="micro")

print("F1 (micro):", f1_micro)
print("F1 (macro):", f1_macro)
print("Precision (micro):", precision_micro)
print("Recall (micro):", recall_micro)


🔍 Evaluation metrics:


e:\Product Comparator\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


F1 (micro): 0.05156382079459002
F1 (macro): 0.0011720500141364143
Precision (micro): 0.40397350993377484
Recall (micro): 0.027539503386004513


In [11]:
# Number of unique labels
num_labels = len(all_labels)
print(f"Total number of unique labels: {num_labels}\n")

# Print the labels themselves (can truncate if too many)
print("Some labels (first 50 shown):")
for i, label in enumerate(all_labels[:50]):
    print(f"{i}: {label}")

# Optionally, save all labels to a file for full inspection
with open("all_labels.txt", "w", encoding="utf-8") as f:
    for label in all_labels:
        f.write(label + "\n")
print("\n✅ All labels saved to all_labels.txt")


Total number of unique labels: 15792

Some labels (first 50 shown):
0: 1-inch sensor
1: 10-bit HDR video recording
2: 100x Space Zoom
3: 100x Zoom
4: 10x optical zoom
5: 10x periscope diffraction spikes
6: 10x periscope zoom
7: 10x zoom lens
8: 10x zoom performance
9: 12-bit DNG option in Pro Mode
10: 120 Hz refresh rate capability
11: 120Hz Display
12: 120Hz Refresh Rate
13: 120Hz app support
14: 120Hz at 1440p resolution
15: 120Hz display
16: 120Hz display smoothness
17: 120Hz dynamic refresh rate
18: 120Hz refresh rate
19: 120Hz refresh rate in Performance mode
20: 120Hz refresh rate mode
21: 120Hz support in apps
22: 128GB Storage capacity
23: 15 Watt Wireless charging
24: 15W Wireless charging
25: 15W wireless charging
26: 16GB RAM availability (regional)
27: 16GB RAM option
28: 200MP periscope camera
29: 200MP sensor
30: 2FA support
31: 2x zoom detail
32: 3.5mm audio jack
33: 3.5mm headphone jack
34: 3.5mm jack
35: 30x zoom detail
36: 3rd party stylus support
37: 3x Telephoto Cam